In [1]:
import sys
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python version: 3.12.10 (v3.12.10:0cc81280367, Apr  8 2025, 08:47:00) [Clang 13.0.0 (clang-1300.0.29.30)]
Pandas version: 3.0.6
NumPy version: 2.5.4


In [2]:
%pip install nba_api

Note: you may need to restart the kernel to use updated packages.


In [3]:
from nba_api.stats.endpoints import leaguedashplayerstats

# Retrieve 2024-25 NBA regular season player statistics
nba_stats = leaguedashplayerstats.LeagueDashPlayerStats(
    season="2024-25",
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame",
    measure_type_detailed_defense="Base",
    timeout=60
)

# Convert the API response into a DataFrame
df = nba_stats.get_data_frames()[0]

print("Number of players:", len(df))
display(df.head())

Number of players: 569


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.0,26,14,12,0.538,...,201,247,207,219,300,292,159,44,281,1
1,1631260,AJ Green,AJ,1610612749,MIL,25.0,73,44,29,0.603,...,520,136,394,278,83,342,281,44,318,1
2,1642358,AJ Johnson,AJ,1610612764,WAS,20.0,29,8,21,0.276,...,120,239,321,273,522,332,281,44,328,2
3,203932,Aaron Gordon,Aaron,1610612743,DEN,29.0,51,33,18,0.647,...,84,277,89,98,26,127,134,44,124,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,28.0,62,39,23,0.629,...,415,413,350,363,142,422,281,44,403,1


In [4]:
# See all available columns
print(df.columns.tolist())

# Check dataset dimensions
print("Shape:", df.shape)

# Check for missing values
display(df.isnull().sum().sort_values(ascending=False).head(15))

# Preview important player statistics
display(df[
    ["PLAYER_NAME", "TEAM_ABBREVIATION", "GP",
     "PTS", "REB", "AST", "STL", "BLK"]
].head(10))

['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT']
Shape: (569, 67)


PLAYER_ID            0
PLAYER_NAME          0
NICKNAME             0
TEAM_ID              0
TEAM_ABBREVIATION    0
AGE                  0
GP                   0
W                    0
L                    0
W_PCT                0
MIN                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
dtype: int64

,PLAYER_NAME,TEAM_ABBREVIATION,GP,PTS,REB,AST,STL,BLK
0,A.J. Lawson,TOR,26,9.1,3.3,1.2,0.5,0.2
1,AJ Green,MIL,73,7.4,2.4,1.5,0.5,0.1
2,AJ Johnson,WAS,29,7.6,2.0,2.6,0.4,0.1
3,Aaron Gordon,DEN,51,14.7,4.8,3.2,0.5,0.3
4,Aaron Holiday,HOU,62,5.5,1.3,1.3,0.3,0.2
5,Aaron Nesmith,IND,45,12.0,4.0,1.2,0.8,0.4
6,Aaron Wiggins,OKC,76,12.0,3.9,1.8,0.8,0.2
7,Adam Flagler,OKC,37,1.8,0.7,0.3,0.2,0.1
8,Adama Sanogo,CHI,4,2.0,1.5,0.3,0.0,0.0
9,Adem Bona,PHI,58,5.8,4.2,0.5,0.4,1.2


In [5]:
from pathlib import Path

# Locate the project folder
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Create the raw data folder
output_dir = project_dir / "data" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

# Save the CSV
output_path = output_dir / "nba_player_stats_2024_25.csv"
df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/charlieellis/nba-player-value-analyzer/data/raw/nba_player_stats_2024_25.csv


## Data Quality Checks

Before analyzing player performance, we verify that
the dataset contains unique players, identify missing
values, and inspect the available statistics.

In [6]:
# Check dataset dimensions
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

# Check for duplicate player IDs
print("Duplicate players:", df["PLAYER_ID"].duplicated().sum())

# Check for missing values
missing = df.isnull().sum()
print("\nColumns with missing values:")
print(missing[missing > 0])

# Check basic statistics
display(df[["GP", "PTS", "REB", "AST", "MIN"]].describe())

Total rows: 569
Total columns: 67
Duplicate players: 0

Columns with missing values:
Series([], dtype: int64)


,GP,PTS,REB,AST,MIN
count,569.000000,569.00000,569.000000,569.000000,569.000000
mean,46.231986,8.88717,3.595606,2.085237,19.428998
std,24.357135,6.56826,2.422532,1.819464,9.539740
min,1.000000,0.00000,0.000000,0.000000,0.900000
25%,25.000000,4.00000,1.800000,0.800000,11.800000
50%,50.000000,7.20000,3.200000,1.500000,19.500000
75%,68.000000,12.10000,4.800000,2.700000,27.200000
max,82.000000,32.70000,13.900000,11.600000,37.700000


In [7]:
# Select columns relevant to our project
columns = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "TEAM_ABBREVIATION",
    "AGE",
    "GP",
    "MIN",
    "PTS",
    "REB",
    "AST",
    "STL",
    "BLK",
    "TOV",
    "FG_PCT",
    "FG3_PCT",
    "FT_PCT",
    "FGM",
    "FGA",
    "FG3M",
    "FG3A",
    "FTM",
    "FTA",
    "TEAM_COUNT"
]

players = df[columns].copy()

# Rename columns for easier analysis
players = players.rename(columns={
    "PLAYER_ID": "player_id",
    "PLAYER_NAME": "player_name",
    "TEAM_ABBREVIATION": "team",
    "AGE": "age",
    "GP": "games_played",
    "MIN": "minutes_per_game",
    "PTS": "points_per_game",
    "REB": "rebounds_per_game",
    "AST": "assists_per_game",
    "STL": "steals_per_game",
    "BLK": "blocks_per_game",
    "TOV": "turnovers_per_game"
})

display(players.head())

,player_id,player_name,team,age,games_played,minutes_per_game,points_per_game,rebounds_per_game,assists_per_game,steals_per_game,...,FG_PCT,FG3_PCT,FT_PCT,FGM,FGA,FG3M,FG3A,FTM,FTA,TEAM_COUNT
0,1630639,A.J. Lawson,TOR,24.0,26,18.7,9.1,3.3,1.2,0.5,...,0.421,0.327,0.683,3.1,7.3,1.3,3.9,1.7,2.4,1
1,1631260,AJ Green,MIL,25.0,73,22.7,7.4,2.4,1.5,0.5,...,0.429,0.427,0.815,2.5,5.8,2.1,5.0,0.3,0.4,1
2,1642358,AJ Johnson,WAS,20.0,29,22.0,7.6,2.0,2.6,0.4,...,0.385,0.267,0.865,2.8,7.3,0.8,3.1,1.1,1.3,2
3,203932,Aaron Gordon,DEN,29.0,51,28.4,14.7,4.8,3.2,0.5,...,0.531,0.436,0.810,5.2,9.7,1.5,3.4,2.8,3.5,1
4,1628988,Aaron Holiday,HOU,28.0,62,12.8,5.5,1.3,1.3,0.3,...,0.437,0.398,0.829,1.9,4.3,1.2,2.9,0.5,0.7,1


In [8]:
# Retrieve season totals for more accurate calculations
nba_totals = leaguedashplayerstats.LeagueDashPlayerStats(
    season="2024-25",
    season_type_all_star="Regular Season",
    per_mode_detailed="Totals",
    measure_type_detailed_defense="Base",
    timeout=60
)

totals = nba_totals.get_data_frames()[0]

display(totals[
    ["PLAYER_NAME", "GP", "MIN", "PTS", "FGA", "FTA"]
].head())

,PLAYER_NAME,GP,MIN,PTS,FGA,FTA
0,A.J. Lawson,26,486.410000,236,190,63
1,AJ Green,73,1659.141667,541,424,27
2,AJ Johnson,29,638.386667,220,213,37
3,Aaron Gordon,51,1446.716667,748,497,179
4,Aaron Holiday,62,792.246667,340,268,41


In [9]:
import numpy as np

# Calculate true shooting percentage using season totals
denominator = 2 * (totals["FGA"] + 0.44 * totals["FTA"])

totals["TS_PCT"] = np.where(
    denominator > 0,
    totals["PTS"] / denominator,
    np.nan
)

# Merge TS% into the cleaned per-game dataset
players = players.merge(
    totals[["PLAYER_ID", "TS_PCT"]],
    left_on="player_id",
    right_on="PLAYER_ID",
    how="left",
    validate="one_to_one"
).drop(columns="PLAYER_ID")

display(
    players[[
        "player_name",
        "points_per_game",
        "minutes_per_game",
        "TS_PCT"
    ]].head(10)
)

,player_name,points_per_game,minutes_per_game,TS_PCT
0,A.J. Lawson,9.1,18.7,0.541981
1,AJ Green,7.4,22.7,0.620584
2,AJ Johnson,7.6,22.0,0.479763
3,Aaron Gordon,14.7,28.4,0.649576
4,Aaron Holiday,5.5,12.8,0.594322
5,Aaron Nesmith,12.0,24.9,0.653066
6,Aaron Wiggins,12.0,22.9,0.595704
7,Adam Flagler,1.8,5.5,0.335467
8,Adama Sanogo,2.0,5.4,0.571429
9,Adem Bona,5.8,15.6,0.713983


In [10]:
from pathlib import Path

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

players.to_csv(
    output_dir / "nba_players_cleaned.csv",
    index=False
)

print("Cleaned player dataset saved successfully!")

Cleaned player dataset saved successfully!
